<a href="https://colab.research.google.com/github/elkins/synth-pdb/blob/main/examples/interactive_tutorials/noe_network_explorer.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# The NOE Network: The Web That Defines a Protein's Shape

NMR protein structures aren't computed from a formula — they're **triangulated** from thousands of
experimental distance measurements called **Nuclear Overhauser Effects (NOEs)**. Each NOE is a
measured signal between two hydrogen atoms that must be within ~5 Å of each other.

Together, these measurements form a dense web of constraints that uniquely pins down the protein's
3D fold. This is how **Kurt Wüthrich** determined the first protein NMR structures in the 1980s,
earning the Nobel Prize in Chemistry in 2002.

> **In this tutorial**, we'll calculate the NOE/contact network for ubiquitin, render it as a
> glowing 3D web on the structure, and analyze the per-residue connectivity patterns.

In [ ]:
import os
import sys

IN_COLAB = "google.colab" in sys.modules
if IN_COLAB:
    # @title Environment Setup
    !pip install -q synth-pdb synth-nmr biotite matplotlib py3Dmol
else:
    sys.path.insert(0, os.path.abspath("../../"))

import io
import numpy as np
import matplotlib
import matplotlib.pyplot as plt
import matplotlib.cm as cm
import biotite.structure.io.pdb as bpdb
import py3Dmol
from synth_pdb.generator import generate_pdb_content
from synth_nmr import calculate_synthetic_noes

print("✅ Environment ready")

## The Physics of the NOE: A Distance Ruler with Steep Fall-off

The NOE signal intensity is proportional to **r⁻⁶** — the inverse sixth power of the
interproton distance:

$$I_{\text{NOE}} \propto \frac{1}{r^6}$$

This steep dependence means:
- **Close protons** (< 3 Å): extremely strong signals
- **Medium contacts** (3–4 Å): moderate signals  
- **Long range** (4–5 Å): weak but detectable
- **Beyond ~5–6 Å**: signal vanishes into noise

In practice, we treat each observed NOE as an **upper distance bound**: *"these two atoms
MUST be within this distance."* A structure that violates any restraint is physically
impossible — this is what makes the network so powerful.

> **Note:** Our synthetic structures lack explicit hydrogen atoms, so we'll build the
> equivalent **Cα contact network** (cutoff 8 Å) as a faithful proxy — this is in fact
> what many computational NMR studies use for long-range restraint analysis.

In [ ]:
# Ubiquitin — 76 residues, one of the most studied NMR proteins
UBIQUITIN_SEQ = "MQIFVKTLTGKTITLEVEPSDTIENVKAKIQDKEGIPPDQQRLIFAGKQLEDGRTLSDYNIQKESTLHLVLRLRGG"

print("Generating ubiquitin structure...")
pdb_str = generate_pdb_content(sequence_str=UBIQUITIN_SEQ, conformation="alpha", seed=42)

struct = bpdb.PDBFile.read(io.StringIO(pdb_str)).get_structure(model=1)
print(f"Structure: {len(UBIQUITIN_SEQ)} residues, {struct.array_length()} atoms")

# --- Build Cα contact network (proxy for NOE long-range restraints) ---
ca = struct[struct.atom_name == "CA"]
n_res = len(ca)
NOE_CUTOFF = 8.0  # Å — includes both medium and long-range contacts

contacts = []
for i in range(n_res):
    for j in range(i + 4, n_res):  # skip i±1/2/3 (trivially bonded neighbours)
        d = float(np.linalg.norm(ca.coord[i] - ca.coord[j]))
        if d <= NOE_CUTOFF:
            contacts.append(
                {
                    "res_i": int(ca.res_id[i]),
                    "res_j": int(ca.res_id[j]),
                    "coord_i": ca.coord[i].tolist(),
                    "coord_j": ca.coord[j].tolist(),
                    "distance": d,
                }
            )

print("\n📊 Contact network statistics:")
print(f"   Total Cα–Cα contacts (|i-j|≥4, d≤{NOE_CUTOFF}Å): {len(contacts)}")
distances = [c["distance"] for c in contacts]
print(f"   Distance range: {min(distances):.2f} – {max(distances):.2f} Å")
print(f"   Mean distance:  {np.mean(distances):.2f} Å")

## Visualizing the Restraint Web in 3D

Each contact is rendered as a **colored cylinder** connecting the two Cα atoms:
- 🔴 **Red** — short contacts (< 6 Å): strong NOE signal, defines the hydrophobic core
- 🟠 **Orange** — medium contacts (6–7 Å): moderate NOE
- 🟡 **Gold** — long-range (7–8 Å): weak but structurally important

The protein backbone is shown as a semi-transparent gray cartoon beneath the web.

In [ ]:
view = py3Dmol.view(width=900, height=520)
view.addModel(pdb_str, "pdb")
view.setStyle({"cartoon": {"color": "#c0c0c0", "opacity": 0.45}})

# Add Cα spheres for visual anchoring
view.addStyle({"atom": "CA"}, {"sphere": {"radius": 0.28, "color": "#e0e0e0", "opacity": 0.6}})

# Add contact cylinders — cap at 400 for performance
displayed = 0
for c in contacts:
    if displayed >= 400:
        break
    d = c["distance"]
    if d < 6.0:
        color, opacity = "#e63946", 0.75
    elif d < 7.0:
        color, opacity = "#f4a261", 0.65
    else:
        color, opacity = "#ffd166", 0.50
    ci, cj = c["coord_i"], c["coord_j"]
    view.addCylinder(
        {
            "start": {"x": ci[0], "y": ci[1], "z": ci[2]},
            "end": {"x": cj[0], "y": cj[1], "z": cj[2]},
            "radius": 0.09,
            "color": color,
            "opacity": opacity,
        }
    )
    displayed += 1

view.setBackgroundColor("#1a1a2e")
view.zoomTo()
print(f"Displaying {displayed} contacts. Rotate with mouse — the protein core glows!")
view.show()

## Per-Residue Connectivity: The Topology of the Core

Residues buried in the hydrophobic core make **many** long-range contacts because they're
surrounded on all sides. Surface residues and loops make fewer. This means the contact count
per residue is itself a **structural reporter** — it tells you which residues are in the core
without ever seeing the 3D structure directly.

In [ ]:
# Count contacts per residue
from collections import defaultdict

count_per_res = defaultdict(int)
for c in contacts:
    count_per_res[c["res_i"]] += 1
    count_per_res[c["res_j"]] += 1

res_ids = sorted(count_per_res.keys())
counts = [count_per_res[r] for r in res_ids]
mean_cnt = np.mean(counts)

# Colour bars by relative connectivity (viridis: high=yellow, low=purple)
norm = matplotlib.colors.Normalize(vmin=min(counts), vmax=max(counts))
colors = [cm.viridis(norm(c)) for c in counts]

fig, ax = plt.subplots(figsize=(14, 4))
bars = ax.bar(res_ids, counts, color=colors, edgecolor="none", width=0.85)
ax.axhline(mean_cnt, color="#ef233c", lw=1.5, ls="--", label=f"Mean = {mean_cnt:.1f} contacts")
ax.fill_between(res_ids, counts, alpha=0.15, color="#4361ee")

sm = cm.ScalarMappable(cmap="viridis", norm=norm)
sm.set_array([])
plt.colorbar(sm, ax=ax, label="Contact count", pad=0.01)

ax.set_xlabel("Residue Number", fontsize=12)
ax.set_ylabel("Number of Long-Range Contacts", fontsize=12)
ax.set_title(
    "Cα Contact Connectivity Per Residue\nHigh connectivity → buried in hydrophobic core",
    fontsize=13,
    fontweight="bold",
)
ax.legend(fontsize=11)
ax.set_xlim(res_ids[0] - 1, res_ids[-1] + 1)
plt.style.use("seaborn-v0_8-whitegrid")
plt.tight_layout()
plt.savefig("noe_connectivity.png", dpi=150, bbox_inches="tight")
plt.show()
print(
    f"Peak connectivity at residue {res_ids[counts.index(max(counts))]} ({max(counts)} contacts) — likely a core residue"
)

## Distance Distribution: The Packing Fingerprint

A well-packed globular protein shows a characteristic **bimodal** distance distribution:
a cluster of short contacts from the hydrophobic core and a broader population from surface
interactions. An IDP would show a flatter, right-shifted distribution — fewer short contacts
because nothing holds the chain together.

In [ ]:
fig, ax = plt.subplots(figsize=(10, 4))

n, bins, patches = ax.hist(distances, bins=30, edgecolor="none", alpha=0.0)

# Colour each bar by position (viridis gradient)
for patch, left in zip(patches, bins[:-1]):
    norm_val = (left - bins[0]) / (bins[-1] - bins[0])
    patch.set_facecolor(cm.plasma(0.2 + 0.6 * norm_val))
    patch.set_alpha(0.85)

# KDE overlay
from scipy.stats import gaussian_kde

kde = gaussian_kde(distances, bw_method=0.25)
x_smooth = np.linspace(bins[0], bins[-1], 200)
kde_vals = kde(x_smooth) * len(distances) * (bins[1] - bins[0])
ax.plot(x_smooth, kde_vals, color="white", lw=2.5, label="KDE")

ax.set_facecolor("#0d1117")
fig.patch.set_facecolor("#0d1117")
ax.tick_params(colors="#cdd6f4")
ax.spines[:].set_color("#313244")
ax.xaxis.label.set_color("#cdd6f4")
ax.yaxis.label.set_color("#cdd6f4")
ax.title.set_color("#cdd6f4")

ax.set_xlabel("Cα–Cα Distance (Å)", fontsize=12)
ax.set_ylabel("Number of Contacts", fontsize=12)
ax.set_title(
    "Distribution of Long-Range Contact Distances\nThe packing fingerprint of the folded core",
    fontsize=13,
    fontweight="bold",
)
ax.legend(fontsize=11, facecolor="#1e1e2e", labelcolor="#cdd6f4")
plt.tight_layout()
plt.savefig("noe_distance_hist.png", dpi=150, bbox_inches="tight")
plt.show()

## Conclusion: From Web to Structure

The contact/NOE network we've explored is not just a visualization trick — it IS the
experimental data. In a real NMR structure determination:

1. You record a **NOESY spectrum** → identify peaks
2. Assign each peak to a pair of residues → this is the hardest step
3. Convert peak intensity → distance bound (via r⁻⁶ calibration)
4. Feed all restraints into **CYANA** or **CNS** → structure calculation
5. The result is an **ensemble** of structures satisfying all restraints

The average NMR structure determination uses **1,000–3,000 NOE restraints** for a
100-residue protein. Each restraint you see as a cylinder in the 3D view above is
one piece of experimental evidence shaping the final structure.

**The density of the web reveals the quality of the NMR data** — a sparse web means
a poorly determined structure. A dense, interconnected web gives high-precision coordinates.